# Session 7: Exercises

**Before session 8.** Predict, then check. Answers go in the **✍️** cells, and code TODOs are marked
`# TODO`.

Exercises 1 to 4 run on a **scripted model**: a stand-in that replays fixed replies and records what
it was shown. They cost nothing and need no key, and that's the point rather than a shortcut: a rail
acts *before* or *around* the model, so you can test it without one. Exercise 5 is the exception,
and says so.

## Setup: given, just run it

In [ ]:
import itertools
import sys
import time
import warnings
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import (HumanInTheLoopMiddleware, PIIMiddleware,
                                         after_agent, before_agent)
from langchain.messages import AIMessage
from langchain_core._api import LangChainBetaWarning
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langgraph.checkpoint.memory import InMemorySaver

warnings.filterwarnings("ignore", category=LangChainBetaWarning)

sys.path.insert(0, str(Path.cwd() / "project"))   # the v4 snapshot
from agent import transcript                       # noqa: E402
from tools import TOOLS                            # noqa: E402
from triage import PROMPTS                         # noqa: E402
import middleware as mw                            # noqa: E402

AYESHA = mw.customer_for("ayesha@example.com")     # owns ORD-5001 and ORD-5002
tickets = itertools.count(1)


class ScriptedModel(GenericFakeChatModel):
    """A stand-in model: replays the replies it's given, and records what it was shown each call."""

    seen: list = []

    def bind_tools(self, tools, **kwargs):
        return self                     # the replies are scripted, so the tools are ignored

    def _generate(self, messages, *args, **kwargs):
        self.seen.append([m.text for m in messages])
        return super()._generate(messages, *args, **kwargs)


def scripted(*replies, forever=None):
    """A ScriptedModel with these replies (or one reply repeated forever)."""
    return ScriptedModel(messages=itertools.repeat(forever) if forever else iter(replies), seen=[])


def call(name, **args):
    """A scripted model turn that asks for one tool."""
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": f"call-{name}",
                                      "type": "tool_call"}])


def make_agent(*middleware, chat_model):
    """The v4 agent (same prompt, same seven tools) with only the rails you pass."""
    return create_agent(model=chat_model, tools=TOOLS, system_prompt=PROMPTS["agent_prompt"],
                        middleware=list(middleware), context_schema=mw.Customer,
                        checkpointer=InMemorySaver())


def ask(agent, text, who=AYESHA, show=True):
    """One message, as `who`, on a new ticket. Returns the result."""
    config = {"configurable": {"thread_id": f"T-{next(tickets)}"}}
    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config, context=who)
    if show:
        print(transcript(result))
    return result


print("ready: every exercise except 5 runs on a scripted model, with no key and no cost")

## ⭐ Exercise 1: Mask, don't redact

**What this teaches:** the PII *strategy* is a product decision too.

v4 redacts card numbers completely (notebook section 5). Switch the card rule to `strategy="mask"`,
keep v4's email rule (`mw.PII_RULES[1]`), and send the same message as section 5.2. A scripted model
is enough, because PII handling happens before the model is called: `model.seen` shows exactly what
it was given.

In [ ]:
MSG = "I'm ayesha@example.com. Card 4111 1111 1111 1111 was charged twice for ORD-5002."
model = scripted(AIMessage("Thanks, Ayesha. I'll look into the double charge."))

# TODO: build an agent with PIIMiddleware("credit_card", strategy="mask") and mw.PII_RULES[1],
#       using chat_model=model. ask(...) it MSG, then print model.seen[0][-1]: what the model saw.

**✍️ Your answer.** What exactly did the model see? Name one support workflow where a person *needs*
the last four digits, and one where even those are too much.

## ⭐⭐ Exercise 2: An off-topic filter, without false positives

**What this teaches:** a deterministic filter is only as good as its test set, and the mistake that
hurts is blocking a *real* customer.

Write `off_topic_filter`, a `@before_agent` rail that ends the run politely on homework requests,
such as *"write my essay"* or *"do my homework"*. It must **not** block legitimate messages that
contain the same words. Test it against all four messages.

In [ ]:
TESTS = [   # (message, should_block)
    ("Can you write my history essay on the Mughal empire?", True),
    ("Please do my homework: solve these five equations.", True),
    ("Can you write down my order number for me? It's ORD-5002.", False),
    ("My son needs headphones for homework calls. Is the AeroSound Pro in stock?", False),
]
OFF_TOPIC_REPLY = "I can only help with ShopWise orders, products and refunds."

# TODO: a phrase list, and a @before_agent(can_jump_to=["end"]) function that returns
#       {"messages": [AIMessage(OFF_TOPIC_REPLY)], "jump_to": "end"} on a match.

# TODO: for each test, ask(make_agent(off_topic_filter, chat_model=scripted(forever=AIMessage("ok"))),
#       message, show=False), check whether the last message is OFF_TOPIC_REPLY, and print PASS or
#       FAIL per test.

**✍️ Your answer.** Did your first phrase list pass all four? Which kind of mistake did it make: a
false positive (a real customer blocked) or a false negative (homework let through)? Which one costs
ShopWise more?

## ⭐⭐⭐ Exercise 3: Only big refunds need a manager

**What this teaches:** human approval has a cost (a person's time, the customer's wait), so real
systems approve small actions automatically.

Build a refund-approval rail that pauses **only when the amount is over 50**, using the `when=`
predicate. It receives the tool-call request and returns `True` to pause (verified on
`langchain` 1.3.14):

```python
{"issue_refund": {"allowed_decisions": ["approve", "reject"],
                  "when": lambda request: request.tool_call["args"]["amount"] > 50}}
```

The scripted model asks for a 30 refund in one run and a 129 refund in the other. The 30 should
go straight through; the 129 should pause.

In [ ]:
def refund_model(amount):
    """A scripted model that asks to refund `amount` on ORD-5001, then answers."""
    return scripted(call("issue_refund", order_id="ORD-5001", amount=amount, reason="damaged case"),
                    AIMessage("Done."))

# TODO: threshold_approval = HumanInTheLoopMiddleware(interrupt_on={...})
# TODO: for amount in (30, 129): run ask(make_agent(threshold_approval, chat_model=refund_model(amount)),
#       "Please refund ORD-5001.", show=False) and print whether "__interrupt__" is in the result,
#       and whether an issue_refund result appears in its messages.

**✍️ Your answer.** Did both behave as intended? Who picks the 50, and where should that number live
in the project (hint: session 3's `config.py`)? What could a customer do to get 129 back past this
rail, and where should the rule that stops it live?

## ⭐⭐ Exercise 4: Which empty reply is which?

**What this teaches:** an empty reply is a symptom, and the right response depends on the cause
(notebook section 8.3).

**Part a.** Six replies a model might send back. For each one, **predict** what
`mw.blocked_reason()` returns (`None`, `output_blocked`, `input_blocked`, `out_of_tokens` or
`empty`). Then run the cell and check.

In [ ]:
REPLIES = {
    "A": AIMessage("Your parcel is with SwiftEx.", response_metadata={"finish_reason": "stop"}),
    "B": AIMessage("", response_metadata={"finish_reason": "tool_calls"},
                   tool_calls=[{"name": "get_order", "args": {"order_id": "ORD-5003"}, "id": "1",
                                "type": "tool_call"}]),
    "C": AIMessage("", response_metadata={"finish_reason": "content_filter",
                                          "native_finish_reason": "SAFETY"}),
    "D": AIMessage("", response_metadata={"finish_reason": "length"}),
    "E": AIMessage("", response_metadata={"finish_reason": "stop"}),
    "F": AIMessage("", response_metadata={"prompt_feedback": {"block_reason": "PROHIBITED_CONTENT"}}),
}

for name, reply in REPLIES.items():
    print(name, mw.blocked_reason(reply))

**Part b.** v4 escalates every empty reply, which is safe but wasteful: a spent output budget is
worth one retry with a bigger budget. Write `next_step(reply)`, returning one of:

| Returns | When |
|---|---|
| `"send"` | a normal reply, or a tool request |
| `"escalate"` | a block, on input or output |
| `"retry with a bigger budget"` | the output cap ran out |
| `"retry once, then escalate"` | empty, and nothing says why |

In [ ]:
EXPECTED = {"A": "send", "B": "send", "C": "escalate", "D": "retry with a bigger budget",
            "E": "retry once, then escalate", "F": "escalate"}

# TODO: def next_step(reply) -> str, built on mw.blocked_reason(reply).
# TODO: for each reply, print PASS or FAIL against EXPECTED.

**✍️ Your answer.** Why is B empty but fine? Why is retrying C the wrong reflex, when retrying D is
the right one? And where would you *apply* `next_step` in v4: which hook?

## ⭐⭐ Exercise 5: [ADV] What does a model-based guardrail cost?

**What this teaches:** "deterministic first, model-based sparingly", with numbers.

> **This one calls a real model.** About a dozen small calls on the default route, roughly a cent.
> Everything above was free.

Write `promise_judge`, an `@after_agent` rail that asks the model *"does this reply promise money, a
replacement, or a legal outcome?"* and, if yes, replaces the reply with an escalation. Then time five
ordinary questions with and without it, and count model calls per reply.

In [ ]:
from pydantic import BaseModel                                  # noqa: E402
from utils import ModelReturnedNothing, ask_json, build_model   # noqa: E402

model = build_model()                                   # the real one, from here on
TANVIR = mw.customer_for("tanvir@example.com")
ESCALATE = "This one needs a person. A member of the ShopWise team will follow up on this ticket."
QUESTIONS = ["Where is my parcel for ORD-5003?", "Is the kettle in stock?",
             "What's the warranty on the AeroSound Pro?", "Has ORD-5003 shipped?",
             "What did I order most recently?"]

# TODO: a pydantic model Verdict(promises: bool), and an @after_agent promise_judge that asks
#       about the last message's text with ask_json(model, Verdict, [("human", ...)]).
#       (Not with_structured_output: on gpt-oss it can return None. Session 4, section 4.)
#       Decide what the judge does when it gets no answer at all.
# TODO: time each question with time.perf_counter(), with and without promise_judge
#       (make_agent(..., chat_model=model)), and print the average seconds per reply.

**✍️ Your answer.** What did the judge add per reply, in seconds and in calls? Multiply by 1,000
tickets a day. When is it worth it?